In [ ]:
%pip install requests beautifulsoup4 lxml selenium pandas webdriver-manager

In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get('https://example.com')

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('body > div > p:nth-child(3)')
print(result.text)

#종료
driver.quit()

Learn more


# 다나와 자동차 판매량

In [18]:
url = 'https://auto.danawa.com/auto/?Work=record'


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get(url)

html = driver.page_source   # 정적에서 response.text 와 동일
soup = BeautifulSoup(html, 'lxml')
result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div > div:nth-child(3) > div.left > table > tbody')
result = result.select('tr')
for tag in result:
    # print(tag.select_one('.num').text)
    print(tag.select_one('a').text.strip(), tag.select_one('.num').text    )

#종료
driver.quit()

기아 42,066
현대 40,066
제네시스 6,942
KGM 3,701
르노코리아 2,000


In [12]:
result.select('tr')

[<tr>
 <td class="rank">1</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=307&amp;Month=2026-02-00">
 <img alt="기아" src="https://autoimg.danawa.com/photo/brand/307_40.png"/>기아
 	                            </a>
 </td>
 <td class="num">42,066</td>
 <td class="share">44.0%</td>
 </tr>,
 <tr>
 <td class="rank">2</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=303&amp;Month=2026-02-00">
 <img alt="현대" src="https://autoimg.danawa.com/photo/brand/303_40.png"/>현대
 	                            </a>
 </td>
 <td class="num">40,066</td>
 <td class="share">41.9%</td>
 </tr>,
 <tr>
 <td class="rank">3</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=304&amp;Month=2026-02-00">
 <img alt="제네시스" src="https://autoimg.danawa.com/photo/brand/304_40.png"/>제네시스
 	                            </a>
 </td>
 <td class="num">6,942</td>
 <td class="share">7.3%</td>
 </tr>,
 <tr>
 <td class="rank">4</td>
 <td class="title">
 <a href="/auto/?Work=record&amp;Brand=326&a

# 다나와 2025년 1월부터 12월까지 국산 자동차 판매대수 수집

https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month=2025-01-00

In [54]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

company, amount = [],[]
year,month = [],[]


# 드라이버 설정
service =  Service(ChromeDriverManager().install() )
driver = webdriver.Chrome(service=service)


for _month in range(1,13):
    _year = '2025'
    _url = f'https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month={_year}-{_month:02}-00'   
    

    # 페이지 열기
    driver.get(_url)
    time.sleep(1)  # 페이지 완전 로딩될때까지 기다림.. 대략 1초면 충분하다고 판단

    html = driver.page_source   # 정적에서 response.text 와 동일
    soup = BeautifulSoup(html, 'lxml')
    result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div:nth-child(1) > table > tbody')
    result = result.select('tr')
    count = len(result)
    for tag in result:        
        # print(tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] )
        co, am =  tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] 
        company.append(co); amount.append(am)
    year += [_year]*count
    month += [_month]*count
        

#종료
driver.quit()

In [60]:
import pandas as pd
data = {
    'year' : year,
    'month' : month,
    'company' : company,
    'amount' : amount
}
df = pd.DataFrame(data)
df

,year,month,company,amount
0,2025,1,기아,"38,412"
1,2025,1,현대,"37,230"
2,2025,1,제네시스,"8,824"
3,2025,1,르노코리아,"2,601"
4,2025,1,KGM,"2,300"
...,...,...,...,...
67,2025,12,기아,"44,831"
68,2025,12,제네시스,"9,680"
69,2025,12,르노코리아,"4,771"
70,2025,12,KGM,"2,659"


# mysql 데이터베이스 적재 하기

In [ ]:
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import os
# 환경변수 읽어오기
load_dotenv()

# 환경변수에서 MySQL 연결 정보 로드
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'car_db'),
    'port': int(os.getenv('DB_PORT', 3306))
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
query = '''
insert into sale_car('year','month','compay','amount') 
values(%s,%s,%s,%s)
'''

for _, row in df.iterrows():
    params = (row.year, row.month,row.company, row.amount)
    cursor.execute(query,params=params)

cursor.close()
conn.close()

In [71]:
for _, row in df.iterrows():
    print(row.year, row.month,row.company, row.amount)

2025 1 기아 38,412
2025 1 현대 37,230
2025 1 제네시스 8,824
2025 1 르노코리아 2,601
2025 1 KGM 2,300
2025 1 쉐보레 1,219
2025 2 현대 46,993
2025 2 기아 46,046
2025 2 제네시스 10,223
2025 2 르노코리아 4,881
2025 2 KGM 2,676
2025 2 쉐보레 1,453
2025 3 현대 52,500
2025 3 기아 50,105
2025 3 제네시스 10,595
2025 3 르노코리아 6,116
2025 3 KGM 3,208
2025 3 쉐보레 1,372
2025 4 현대 55,805
2025 4 기아 51,085
2025 4 제네시스 11,504
2025 4 르노코리아 5,252
2025 4 KGM 3,546
2025 4 쉐보레 1,298
2025 5 현대 49,449
2025 5 기아 45,125
2025 5 제네시스 9,517
2025 5 르노코리아 4,202
2025 5 KGM 3,560
2025 5 쉐보레 1,383
2025 6 현대 51,610
2025 6 기아 46,325
2025 6 제네시스 10,454
2025 6 르노코리아 5,013
2025 6 KGM 3,041
2025 6 쉐보레 1,266
2025 7 현대 48,000
2025 7 기아 45,133
2025 7 제네시스 8,227
2025 7 KGM 4,456
2025 7 르노코리아 4,000
2025 7 쉐보레 1,205
2025 8 현대 49,019
2025 8 기아 43,675
2025 8 제네시스 9,311
2025 8 KGM 4,055
2025 8 르노코리아 3,868
2025 8 쉐보레 1,181
2025 9 현대 56,463
2025 9 기아 49,201
2025 9 제네시스 9,538
2025 9 르노코리아 4,182
2025 9 KGM 4,100
2025 9 쉐보레 1,208
2025 10 현대 44,762
2025 10 기아 40,344
2025 10 제네시스 9,